# Region Merge and Clustering

## Load regions

In [ ]:
import pandas as pd
import os

DIR_WSPS = "/home/jupyter/workspace/ws_files"
DIR_ORIG = f"{DIR_WSPS}/Original_Files"

# Canonical location for the deduplicated (union) HAR list - the region
# extractor, burden test and case-control notebooks all read it from here.
DIR_HARS_REF = f"{DIR_WSPS}/HARS_files/HARs_merged"

HAR_LIST = f"{DIR_ORIG}/HAR_list_phase_1.tsv"

har = pd.read_csv(HAR_LIST, sep='\t', header=None, names=['chrom','start','end','name'])
har['chrom'] = har['chrom'].astype(str).str.replace('chr', '', regex=False)

print(f"Original HARs: {len(har)}")

## Cluster by overlap


In [ ]:
har = har.sort_values(['chrom','start']).reset_index(drop=True)
har['cluster'] = 0

cur_cluster = 0
prev_chrom = None
running_end = -1

for i, row in har.iterrows():
    if row['chrom'] != prev_chrom:
        cur_cluster += 1
        running_end = row['end']
    elif row['start'] > running_end:
        cur_cluster += 1
        running_end = row['end']
    else:
        running_end = max(running_end, row['end'])
    har.at[i, 'cluster'] = cur_cluster
    prev_chrom = row['chrom']

n_clusters = har['cluster'].nunique()
print(f"Clusters: {n_clusters}")

## Build the union list

Each cluster collapses into one composite region (min `start`, max `end` across its members), with a composite name that keeps track of the original regions folded into it.

In [ ]:
def make_composite_name(group):
    """cluster{N}-{HAR1}-{HAR2}-... , longest member first"""
    cluster_id = group['cluster'].iloc[0]
    members = group.assign(length=group['end'] - group['start']) \
                   .sort_values('length', ascending=False)['name'].tolist()
    if len(members) > 5:
        composite = f"cluster{cluster_id:05d}-" + "-".join(members[:5]) + f"-plus{len(members)-5}"
    else:
        composite = f"cluster{cluster_id:05d}-" + "-".join(members)
    return composite

union_list = har.groupby('cluster').agg(
    chrom=('chrom', 'first'),
    start=('start', 'min'),
    end=('end', 'max'),
).reset_index()

composite_names = har.groupby('cluster').apply(make_composite_name).reset_index()
composite_names.columns = ['cluster', 'name']

union_list = union_list.merge(composite_names, on='cluster')
union_list = union_list[['chrom','start','end','name','cluster']]
union_list['length_union'] = union_list['end'] - union_list['start']

## Summary

In [ ]:
redundancy = (1 - n_clusters / len(har)) * 100
print(f"{len(har)} HARs -> {n_clusters} clusters ({redundancy:.1f}% redundant)")
print(f"Mean length: {(har['end']-har['start']).mean():.0f} bp (original) vs {union_list['length_union'].mean():.0f} bp (union)")

## Save results

In [ ]:
os.makedirs(DIR_HARS_REF, exist_ok=True)

OUT_UNION = f"{DIR_HARS_REF}/HAR_list_phase_1_union.tsv"
union_list[['chrom','start','end','name']].to_csv(
    OUT_UNION, sep='\t', header=False, index=False
)
print(f"Union list saved to {OUT_UNION}")

OUT_MAP = f"{DIR_HARS_REF}/HAR_cluster_mapping.tsv"
har[['cluster','name','chrom','start','end']].to_csv(OUT_MAP, sep='\t', index=False)
print(f"Cluster mapping saved to {OUT_MAP}")